# Module 16: Logistic Regression and Classification

**Case:** Which characteristics are associated with remaining healthy over the
next survey wave?

We distinguish probability modeling from classification. The case follows
Gábor Békés and Gábor Kézdi's SHARE health materials: start with proportions,
compare the linear probability model and logit, then evaluate probability
predictions and decision thresholds.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    ConfusionMatrixDisplay, brier_score_loss, classification_report,
    confusion_matrix, log_loss, roc_auc_score, roc_curve,
)
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")

**Project data folder.** This setup finds the course folder whether the
notebook starts in the project root or inside a module. `data_dir` points
to the local files prepared by `scripts/fetch_data.py`.

In [ ]:
from pathlib import Path

project_root = Path.cwd()  # Start in the notebook's current working folder.
while not (project_root / "scripts" / "fetch_data.py").is_file():
    if project_root == project_root.parent:
        raise FileNotFoundError("Open this notebook inside the Coding 2 repository.")
    project_root = project_root.parent  # Look one folder higher.

data_dir = project_root / "data" / "raw"

In [ ]:
share = pd.read_csv(data_dir / "share_health_course.csv")
variables = ["stayshealthy", "smoking", "ever_smoked", "female", "age", "income10", "eduyears", "bmi", "exerc"]
share[variables].describe().T.round(2)

## 1. Begin with observed probabilities

In [ ]:
observed = share.groupby("smoking")["stayshealthy"].agg(["mean", "count"])
observed.index = observed.index.map({0: "Not currently smoking", 1: "Currently smoking"})
observed.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=share, x="smoking", y="stayshealthy", errorbar=("ci", 95), ax=ax)
ax.set(title="Observed probability of staying healthy", xlabel="Current smoker", ylabel="Proportion")
plt.show()

## 2. Linear probability model as a transparent baseline

In [ ]:
lpm = smf.ols(
    "stayshealthy ~ smoking + ever_smoked + female + age + income10 + eduyears + bmi + exerc",
    data=share,
).fit(cov_type="HC3")
lpm.summary()

In [ ]:
lpm_predictions = lpm.predict(share)
pd.Series(
    {"minimum": lpm_predictions.min(), "maximum": lpm_predictions.max(), "outside_0_1": ((lpm_predictions < 0) | (lpm_predictions > 1)).sum()}
).round(3)

The LPM coefficient is a change in probability units, but the model can predict
outside zero and one and assumes constant marginal effects. Logit constrains
predictions to the probability scale.

## 3. Separate fitting from evaluation

In [ ]:
train, test = train_test_split(
    share, test_size=0.25, random_state=42, stratify=share["stayshealthy"]
)
train.shape, test.shape

In [ ]:
formula = "stayshealthy ~ smoking + ever_smoked + female + age + income10 + eduyears + bmi + exerc"
logit = smf.logit(formula, data=train).fit(disp=False)
logit.summary()

In [ ]:
odds_ratios = pd.DataFrame(
    {"odds_ratio": np.exp(logit.params), "ci_low": np.exp(logit.conf_int()[0]), "ci_high": np.exp(logit.conf_int()[1])}
)
odds_ratios.loc[["smoking", "ever_smoked", "female", "age", "eduyears"]].round(3)

In [ ]:
marginal_effects = logit.get_margeff(at="overall").summary_frame()
marginal_effects.head(8).round(4)

## 4. Evaluate probabilities before choosing a threshold

In [ ]:
test = test.copy()
test["probability"] = logit.predict(test)
probability_metrics = pd.Series(
    {
        "Brier score": brier_score_loss(test["stayshealthy"], test["probability"]),
        "Log loss": log_loss(test["stayshealthy"], test["probability"]),
        "ROC AUC": roc_auc_score(test["stayshealthy"], test["probability"]),
    }
)
probability_metrics.round(3)

In [ ]:
fraction_positive, mean_predicted = calibration_curve(
    test["stayshealthy"], test["probability"], n_bins=8, strategy="quantile"
)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(mean_predicted, fraction_positive, marker="o", label="Logit")
ax.plot([0, 1], [0, 1], linestyle="--", color="black", label="Perfect calibration")
ax.set(xlabel="Mean predicted probability", ylabel="Observed proportion", title="Calibration on test data")
ax.legend()
plt.show()

## 5. Classification requires a decision threshold

In [ ]:
threshold = 0.5
test["predicted_class"] = (test["probability"] >= threshold).astype(int)
matrix = confusion_matrix(test["stayshealthy"], test["predicted_class"])
matrix

In [ ]:
ConfusionMatrixDisplay(matrix).plot(cmap="Blues")
plt.title("Confusion matrix at threshold 0.50")
plt.show()

In [ ]:
pd.DataFrame(
    classification_report(test["stayshealthy"], test["predicted_class"], output_dict=True)
).T.round(3)

In [ ]:
threshold_rows = []
for cutoff in [0.3, 0.5, 0.7]:
    predicted = (test["probability"] >= cutoff).astype(int)
    tn, fp, fn, tp = confusion_matrix(test["stayshealthy"], predicted).ravel()
    threshold_rows.append(
        {"threshold": cutoff, "false_positive_rate": fp / (fp + tn), "true_positive_rate": tp / (tp + fn)}
    )
pd.DataFrame(threshold_rows).round(3)

In [ ]:
false_positive_rate, true_positive_rate, _ = roc_curve(test["stayshealthy"], test["probability"])
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(false_positive_rate, true_positive_rate)
ax.plot([0, 1], [0, 1], linestyle="--", color="black")
ax.set(xlabel="False-positive rate", ylabel="True-positive rate", title="ROC curve")
plt.show()

## Session 3 reconstruction handoff

Open `prompts/session03-binary-outcomes-reconstruction.md`. Compare whether the
generated workflow keeps probability evaluation separate from classification,
uses the correct outcome in each metric, and distinguishes coefficients, odds
ratios, marginal effects, and predicted probabilities.

### Optional extension

Assign different costs to false positives and false negatives, calculate the
total cost at several thresholds, and select a threshold for that decision—not
because 0.50 is conventional.